# Generate Trajectories

In [7]:
import importlib
import src.core

importlib.reload(src.core)
from src.core import *

In [17]:
from src.core import simulate
from src.plotting import plot_results
import numpy as np

# ----------------------------
# Parameters
# ----------------------------
M = 2.0      # slab mass
m = 1.0      # rod mass
l = 1.0      # rod length
g = 9.81
H = 25       # Horizon


In [ ]:
from src.core import gen_max_theta_data

num_points = 1000
t_span = (0, 2)
n_repeats = 100
sigma = 1.0
theta_max = np.pi / 20
t_all, X_all, F_all = gen_max_theta_data(M, m, g, l, sigma=sigma, theta_max=theta_max, t_span=t_span,
                                          num_points=num_points, n_repeats=n_repeats)

Trajectory Generation: 100%|██████████| 100/100 [01:19<00:00,  1.25it/s]


In [15]:
X_all[0].shape, F_all[0].shape

((4, 226), (226,))

# Form Trajectories

In [20]:
import torch

from torch import nn
from src.core import _physical_state_scale

# Normalize physical data
t0, state_scale, mg = _physical_state_scale(m, g, l)

W = []

for X, F in zip(X_all, F_all):
    Xn = X / state_scale.reshape(4, 1)
    un = F / mg

    T = Xn.shape[1]

    for k in range(T - H - 1):
        x_seq = Xn[:, k:k+H+1].T          # (H+1, 4)
        u_seq = un[k:k+H].reshape(H, 1)   # (H, 1)

        w = np.concatenate([
            x_seq.reshape(-1),
            u_seq.reshape(-1),
        ])

        W.append(w)

W = torch.tensor(np.stack(W), dtype=torch.float32)

print(W.shape)

torch.Size([17439, 129])


# Learn Manifold

In [25]:
from pathlib import Path

from src.manifold_control import BehaviorDecoder

alpha_dim = 8

# -----------------------
# 2. Train manifold decoder
# -----------------------
w_dim = W.shape[1]

decoder = BehaviorDecoder(
    alpha_dim=alpha_dim,
    w_dim=w_dim,
    hidden_dims=(128, 128, 128),
    activation=nn.Tanh,
)

N = W.shape[0]
alpha_table = nn.Parameter(0.1 * torch.randn(N, alpha_dim))

optimizer = torch.optim.Adam(
    list(decoder.parameters()) + [alpha_table],
    lr=1e-3,
)

lambda_alpha = 1e-4

for epoch in range(3000):
    optimizer.zero_grad()

    W_hat = decoder(alpha_table)
    recon_loss = torch.mean((W - W_hat) ** 2)
    alpha_reg = torch.mean(alpha_table ** 2)

    loss = recon_loss + lambda_alpha * alpha_reg

    loss.backward()
    optimizer.step()

    if epoch % 250 == 0:
        print(
            f"epoch={epoch:04d}, "
            f"loss={loss.item():.6f}, "
            f"recon={recon_loss.item():.6f}"
        )

save_dir = Path("saves") / "saved_models"
save_dir.mkdir(parents=True, exist_ok=True)
torch.save(decoder.state_dict(), save_dir / "behavior_decoder_inverse_pendulum.pt")

epoch=0000, loss=0.343183, recon=0.343182
epoch=0250, loss=0.171812, recon=0.171811
epoch=0500, loss=0.132972, recon=0.132971
epoch=0750, loss=0.120172, recon=0.120171
epoch=1000, loss=0.116554, recon=0.116553
epoch=1250, loss=0.104594, recon=0.104593
epoch=1500, loss=0.091761, recon=0.091759
epoch=1750, loss=0.088253, recon=0.088251
epoch=2000, loss=0.085876, recon=0.085874
epoch=2250, loss=0.083270, recon=0.083268
epoch=2500, loss=0.081122, recon=0.081120
epoch=2750, loss=0.079215, recon=0.079213


# Control

## Single Step Test

In [26]:
from src.manifold_control import BehaviorManifoldControlSolver

# -----------------------
# 3. Freeze decoder and solve control problem
# -----------------------
for p in decoder.parameters():
    p.requires_grad_(False)

Q = torch.diag(torch.tensor([1.0, 1.0, 80.0, 10.0]))
R = torch.tensor([[0.1]])

x_ref = torch.zeros(4)
u_ref = torch.zeros(1)

solver = BehaviorManifoldControlSolver(
    decoder=decoder,
    x_dim=4,
    u_dim=1,
    horizon=H,
    Q=Q,
    R=R,
    x_ref=x_ref,
    u_ref=u_ref,
    lambda_theta=100.0,
    lambda_curvature=0.0,
    lr=1e-2,
    max_iter=2000,
    u_bounds=(-10.0, 10.0),
    curvature_mode="none",
)

x0 = torch.tensor([0.0, 0.0, 0.08, 0.0])

x_init = torch.zeros(H + 1, 4)
x_init[0] = x0

u_init = torch.zeros(H, 1)
alpha_init = torch.zeros(alpha_dim)

solution = solver.solve(
    x_init=x_init,
    u_init=u_init,
    alpha_init=alpha_init,
    freeze={
        "theta": True,
        "x": False,
        "u": False,
        "alpha": False,
    },
)

u_plan = solution.u.detach().numpy()
x_plan = solution.x.detach().numpy()

print(solution.loss_dict)
print("first control:", u_plan[0, 0])

{'qr': 0.6214438080787659, 'fit': 0.0031702262349426746, 'curvature': 0.0, 'total': 0.9384664297103882}
first control: 0.20920382


## Small Angle Inverted Pendulum

In [27]:
import numpy as np
import torch

from src.core import wrap_u_caller_as_physical_F_caller
from src.manifold_control import BehaviorManifoldControlSolver


def manifold_u_caller(
    decoder,
    M,
    H=25,
    dt=0.05,
    x_dim=4,
    u_dim=1,
    alpha_dim=8,
    Q=None,
    R=None,
    x_ref=None,
    u_ref=None,
    umax=10.0,
    lambda_theta=100.0,
    lr=1e-2,
    max_iter=1000,
):
    """
    Returns u_caller(t, y), where y is normalized state and u is normalized input.
    """

    device = next(decoder.parameters()).device

    if Q is None:
        Q = torch.diag(torch.tensor([1.0, 1.0, 80.0, 10.0], device=device))
    if R is None:
        R = torch.tensor([[0.1]], device=device)
    if x_ref is None:
        x_ref = torch.zeros(x_dim, device=device)
    else:
        x_ref = torch.as_tensor(x_ref, dtype=torch.float32, device=device)

    if u_ref is None:
        u_ref = torch.zeros(u_dim, device=device)
    else:
        u_ref = torch.as_tensor(u_ref, dtype=torch.float32, device=device)

    for p in decoder.parameters():
        p.requires_grad_(False)

    state = {
        "next_update_t": None,
        "u_seq": np.zeros((H, u_dim)),
        "x_seq": None,
        "alpha": torch.zeros(alpha_dim, device=device),
        "current_u": 0.0,
    }

    def u_caller(t, y):
        y = np.asarray(y, dtype=float).reshape(x_dim)

        if state["next_update_t"] is None or t >= state["next_update_t"] - 1e-12:
            x_init = torch.zeros(H + 1, x_dim, device=device)
            x_init[0] = torch.tensor(y, dtype=torch.float32, device=device)

            # warm start predicted states if available
            if state["x_seq"] is not None:
                x_prev = state["x_seq"]
                x_init[:-1] = torch.tensor(x_prev[1:], dtype=torch.float32, device=device)
                x_init[-1] = x_init[-2]

            u_init = torch.tensor(state["u_seq"], dtype=torch.float32, device=device)
            alpha_init = state["alpha"].detach().clone()

            solver = BehaviorManifoldControlSolver(
                decoder=decoder,
                x_dim=x_dim,
                u_dim=u_dim,
                horizon=H,
                Q=Q,
                R=R,
                x_ref=x_ref,
                u_ref=u_ref,
                lambda_theta=lambda_theta,
                lambda_curvature=0.0,
                lr=lr,
                max_iter=max_iter,
                u_bounds=(-umax, umax),
                curvature_mode="none",
                device=device,
            )

            sol = solver.solve(
                x_init=x_init,
                u_init=u_init,
                alpha_init=alpha_init,
                freeze={
                    "theta": True,
                    "x": False,
                    "u": False,
                    "alpha": False,
                },
            )

            u_opt = sol.u.detach().cpu().numpy()
            x_opt = sol.x.detach().cpu().numpy()

            state["current_u"] = float(u_opt[0, 0])
            state["u_seq"] = np.vstack([u_opt[1:], u_opt[-1:]])
            state["x_seq"] = x_opt
            state["alpha"] = sol.alpha.detach().clone()
            state["next_update_t"] = t + dt

        return float(np.clip(state["current_u"], -umax, umax))

    return u_caller


def manifold_F_caller(
    decoder,
    M,
    m,
    g,
    l,
    H=25,
    dt=0.05,
    x_ref=None,
    u_ref=0.0,
    umax=10.0,
    **kwargs,
):
    """
    Returns F_caller(t, y_phys), matching the physical-unit interface expected by simulate().
    """

    u_caller = manifold_u_caller(
        decoder=decoder,
        M=M / m,
        H=H,
        dt=dt / np.sqrt(l / g),
        x_ref=None if x_ref is None else np.asarray(x_ref) / np.array(
            [l, l / np.sqrt(l / g), 1.0, 1.0 / np.sqrt(l / g)]
        ),
        u_ref=np.array([u_ref / (m * g)]),
        umax=umax / (m * g),
        **kwargs,
    )

    return wrap_u_caller_as_physical_F_caller(u_caller, m, g, l)

In [ ]:
from src.core import simulate
from src.manifold_control import BehaviorDecoder

save_dir = Path("saves") / "saved_models"
save_path = save_dir / "behavior_decoder_inverse_pendulum.pt"
decoder = BehaviorDecoder(
    alpha_dim=alpha_dim,
    w_dim=w_dim,
    hidden_dims=(128, 128, 128),
    activation=nn.Tanh,
)
decoder.load_state_dict(torch.load(save_path))
decoder.eval()

F = manifold_F_caller(
    decoder=decoder,
    M=1.0,
    m=1.0,
    g=9.81,
    l=1.0,
    H=25,
    dt=0.05,
    umax=20.0,
    max_iter=1000,
)

t, x, x_dot, theta, theta_dot, F_seq = simulate(
    F=F,
    M=M,
    m=m,
    g=g,
    l=l,
    y0=[0.0, 0.0, 0.08, 0.0],
    t_span=(0, 5),
    num_points=100,
    method="rk4",
)

# Plotting

In [ ]:

plot_results(t, x, x_dot, theta, theta_dot, F)

In [ ]:
from IPython.display import display, HTML
from src.plotting import animate_point_mass

ani = animate_point_mass(t, x, theta, l, F)
HTML(ani.to_jshtml())

In [ ]:
import torch
from src.manifold_control import BehaviorDecoder, BehaviorManifoldControlSolver, build_w

H = 20
x_dim = 4
u_dim = 1
alpha_dim = 8
w_dim = (H + 1) * x_dim + H * u_dim

decoder = BehaviorDecoder(alpha_dim=alpha_dim, w_dim=w_dim)

solver = BehaviorManifoldControlSolver(
    decoder=decoder,
    x_dim=x_dim,
    u_dim=u_dim,
    horizon=H,
    Q=torch.diag(torch.tensor([1.0, 1.0, 80.0, 12.0])),
    R=torch.eye(u_dim) * 0.1,
    lambda_theta=1.0,
    lambda_curvature=1e-3,
    max_iter=1000,
    lr=1e-3,
)